# Notebook 166: 現代LLMの内部 — スケーリング則と効率化技術

## Modern LLM Internals: Scaling Laws and Efficiency Techniques

---

### このノートブックの位置づけ

**言語モデリングシリーズ** の第7章として、GPT（164）で学んだ基本的なTransformerアーキテクチャから、現代のLLM（LLaMA, Mistral等）で使われる最新技術への進化を学びます。

| 項目 | 内容 |
|------|------|
| 難易度 | ★★★★☆ |
| 所要時間 | 90〜120分 |
| カテゴリ | 言語モデル |

### 学習目標

- [ ] Scaling Laws（Kaplan → Chinchilla）の知見を説明できる
- [ ] RoPE（回転位置エンコーディング）の数理を理解・実装できる
- [ ] GQA（Grouped-Query Attention）の仕組みとMHA/MQAとの違いを説明できる
- [ ] KV-Cacheの推論高速化メカニズムを図解・実装できる
- [ ] FlashAttentionの概念（メモリ効率化）を説明できる
- [ ] RMSNorm, SwiGLUなど現代的コンポーネントを実装できる

### 前提知識

- ✅ Notebook 162（NLPのためのTransformer）
- ✅ Notebook 164（GPT自己回帰LM）

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [GPTから現代LLMへの進化](#2-gptから現代llmへの進化)
3. [Scaling Laws（スケーリング則）](#3-scaling-lawsスケーリング則)
4. [RoPE（回転位置エンコーディング）](#4-rope回転位置エンコーディング)
5. [MHA → MQA → GQA の系譜](#5-mha--mqa--gqa-の系譜)
6. [KV-Cache（推論高速化）](#6-kv-cache推論高速化)
7. [FlashAttention（概念）](#7-flashattention概念)
8. [RMSNorm](#8-rmsnorm)
9. [SwiGLU](#9-swiglu)
10. [モデル並列化の概要](#10-モデル並列化の概要)
11. [現代LLMアーキテクチャの統合](#11-現代llmアーキテクチャの統合)
12. [まとめ・チートシート・よくある間違い・自己評価クイズ](#12-まとめ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import warnings

warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 日本語フォント設定
# ------------------------------------------------------------
def setup_japanese_font():
    """日本語フォントをmatplotlibに設定する関数"""
    font_candidates = ['Hiragino Sans', 'Arial Unicode MS', 'IPAGothic', 'MS Gothic', 'sans-serif']
    plt.rcParams['font.family'] = font_candidates
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (10, 6)
    plt.rcParams['font.size'] = 11
    print("✅ 日本語フォント設定完了")

setup_japanese_font()

# ------------------------------------------------------------
# 再現性のためのシード
# ------------------------------------------------------------
np.random.seed(42)
torch.manual_seed(42)

sns.set_style('whitegrid')

print("✅ 環境セットアップ完了")
print(f"   PyTorch version: {torch.__version__}")

---

## 2. GPTから現代LLMへの進化

### 2.1 LLMの系譜

```
GPT-1 (2018)     → GPT-2 (2019)      → GPT-3 (2020)
117M params        1.5B params          175B params
                                            │
                                            ├─→ InstructGPT / ChatGPT (2022)
                                            ├─→ GPT-4 (2023)
                                            │
BERT (2018) ──→ RoBERTa ──→ DeBERTa
                                            │
LLaMA (2023) → LLaMA 2 (2023) → LLaMA 3 (2024)
  7-65B          7-70B             8-405B
  │
  ├─→ Mistral 7B (2023)  → Mixtral (MoE)
  ├─→ Vicuna, Alpaca
  └─→ Code Llama
```

### 2.2 主要な技術革新

| 時期 | 技術 | 導入モデル | 効果 |
|------|------|----------|------|
| 2017 | Sinusoidal PE | Transformer | 位置情報の注入 |
| 2019 | Pre-LN | GPT-2 | 訓練安定性 |
| 2021 | RoPE | RoFormer | 相対位置エンコーディング |
| 2022 | FlashAttention | 各種 | メモリ効率化 |
| 2023 | GQA | LLaMA 2 | 推論高速化 |
| 2023 | SwiGLU + RMSNorm | LLaMA | 性能向上 |
| 2023 | Sliding Window Attn | Mistral | 長文対応 |

---

## 3. Scaling Laws（スケーリング則）

### 3.1 Kaplan et al. (2020) のスケーリング則

**発見**: 言語モデルの損失 $L$ は、モデルパラメータ数 $N$、データ量 $D$、計算量 $C$ のそれぞれに対してべき乗則（Power Law）に従う：

$$L(N) \propto N^{-\alpha_N}, \quad L(D) \propto D^{-\alpha_D}, \quad L(C) \propto C^{-\alpha_C}$$

ここで $\alpha_N \approx 0.076$、$\alpha_D \approx 0.095$、$\alpha_C \approx 0.050$。

### 3.2 Chinchilla (Hoffmann et al., 2022) の修正

**Kaplanの結論**: 計算予算が限られるなら、モデルを大きくする方がデータを増やすより効率的。

**Chinchillaの修正**: パラメータ数 $N$ とトークン数 $D$ は**同じ比率**で増やすべき。
- 最適比率: $D \approx 20N$（トークン数はパラメータ数の約20倍）
- Chinchilla (70B) は Gopher (280B) と同等以上の性能を、より少ない計算で達成

In [ ]:
# ============================================================
# Section 3: Scaling Laws の可視化
# ============================================================

# --- Kaplan スケーリング則のシミュレーション ---
# L(N) = a * N^(-alpha) + L_inf
# 実際の値に近い定数を使用
alpha_N = 0.076
a_N = 8.0
L_inf = 1.69  # 不可逆損失（entropy of natural language）

# パラメータ数の範囲（100K → 1T）
param_counts = np.logspace(5, 12, 100)
loss_vs_params = a_N * param_counts ** (-alpha_N) + L_inf

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左: パラメータ数 vs 損失 ---
ax = axes[0]
ax.loglog(param_counts, loss_vs_params, linewidth=2.5, color='#e74c3c')

# 代表的なモデルをプロット
models = {
    'GPT-2 (124M)': (124e6, a_N * (124e6)**(-alpha_N) + L_inf),
    'GPT-2 (1.5B)': (1.5e9, a_N * (1.5e9)**(-alpha_N) + L_inf),
    'GPT-3 (175B)': (175e9, a_N * (175e9)**(-alpha_N) + L_inf),
    'LLaMA (65B)': (65e9, a_N * (65e9)**(-alpha_N) + L_inf),
}

for name, (n, l) in models.items():
    ax.scatter(n, l, s=100, zorder=5, edgecolors='black', linewidth=1)
    ax.annotate(name, (n, l), textcoords='offset points', xytext=(10, 5),
                fontsize=9, fontweight='bold')

ax.set_xlabel('パラメータ数 N', fontsize=12)
ax.set_ylabel('テスト損失 L(N)', fontsize=12)
ax.set_title('Scaling Law: パラメータ数 vs 損失\n（Kaplan et al., 2020）', fontsize=13)
ax.grid(True, alpha=0.3, which='both')
ax.axhline(y=L_inf, color='gray', linestyle='--', alpha=0.5, label=f'$L_\\infty$ = {L_inf}')
ax.legend(fontsize=10)

# --- 右: Chinchilla 最適比率 ---
ax = axes[1]

# 計算予算（FLOP）ごとの最適パラメータ数とトークン数
compute_budgets = np.logspace(18, 25, 50)  # 1e18 ~ 1e25 FLOPs

# Kaplan: より大きなモデルを推奨
optimal_N_kaplan = compute_budgets ** 0.73 * 1e-13
optimal_D_kaplan = compute_budgets / (6 * optimal_N_kaplan)

# Chinchilla: N と D を均等に増やす
optimal_N_chinchilla = compute_budgets ** 0.50 * 1e-8
optimal_D_chinchilla = 20 * optimal_N_chinchilla

ax.loglog(optimal_N_kaplan, optimal_D_kaplan, linewidth=2.5, color='#3498db',
          label='Kaplan（モデル重視）', linestyle='--')
ax.loglog(optimal_N_chinchilla, optimal_D_chinchilla, linewidth=2.5, color='#e74c3c',
          label='Chinchilla（均等配分）')

# 20N の参照線
N_ref = np.logspace(7, 12, 50)
ax.loglog(N_ref, 20 * N_ref, color='gray', linestyle=':', alpha=0.5, label='D = 20N')

ax.set_xlabel('最適パラメータ数 N', fontsize=12)
ax.set_ylabel('最適トークン数 D', fontsize=12)
ax.set_title('計算予算配分の比較\nKaplan vs Chinchilla', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("【Scaling Laws の要点】")
print("・モデル性能はパラメータ数/データ量/計算量のべき乗に従って改善する")
print("・Chinchilla則: パラメータ数Nに対してトークン数D≈20Nが最適")
print("・計算予算が限られる場合、巨大モデル+少データよりも適切サイズ+十分データが効率的")

---

## 4. RoPE（回転位置エンコーディング）

### 4.1 なぜ Sinusoidal PE から RoPE へ？

| 手法 | 絶対/相対 | 長さ外挿 | 実装場所 |
|------|----------|---------|--------|
| Sinusoidal PE | 絶対 | △（訓練長に制限） | 入力に加算 |
| Learned PE | 絶対 | ✗（固定長） | 入力に加算 |
| **RoPE** | **相対** | **○（外挿可能）** | **Q, Kに回転を適用** |

### 4.2 RoPE の数理

RoPE の核心は、位置 $m$ のクエリ $\mathbf{q}$ に回転行列を適用すること：

$$\text{RoPE}(\mathbf{q}_m) = R(m\theta) \mathbf{q}_m$$

2次元ブロックでは：
$$R(m\theta_i) = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}$$

ここで $\theta_i = 10000^{-2i/d}$（次元ごとに異なる周波数）。

**重要な性質**: $\mathbf{q}_m^\top \mathbf{k}_n$ の内積が $m - n$（相対位置）のみに依存する。

In [ ]:
# ============================================================
# Section 4: RoPE の実装
# ============================================================

def precompute_rope_frequencies(dim, max_seq_len, base=10000.0):
    """
    RoPE の回転周波数を事前計算する。
    
    Parameters:
        dim: 埋め込み次元（偶数）
        max_seq_len: 最大シーケンス長
        base: 周波数ベース（デフォルト10000）
    Returns:
        freqs: (max_seq_len, dim//2) の周波数テンソル
    """
    # θ_i = base^(-2i/d) for i = 0, 1, ..., d/2 - 1
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    
    # 各位置 m に対する角度 m * θ_i
    positions = torch.arange(max_seq_len).float()
    # 外積: (max_seq_len,) x (dim//2,) → (max_seq_len, dim//2)
    angles = torch.outer(positions, inv_freq)
    
    return angles


def apply_rope(x, angles):
    """
    RoPE を適用する。
    
    Parameters:
        x: (batch, seq_len, n_heads, dim) のテンソル
        angles: (seq_len, dim//2) の角度テンソル
    Returns:
        x_rotated: 回転が適用されたテンソル
    """
    # xを2つに分割: (偶数次元, 奇数次元)
    x_even = x[..., 0::2]  # (..., dim//2)
    x_odd = x[..., 1::2]   # (..., dim//2)
    
    # 角度を適切な形状にブロードキャスト
    cos_angles = torch.cos(angles).unsqueeze(0).unsqueeze(2)  # (1, seq_len, 1, dim//2)
    sin_angles = torch.sin(angles).unsqueeze(0).unsqueeze(2)
    
    # 回転の適用: [cos, -sin; sin, cos] * [even; odd]
    x_rotated_even = x_even * cos_angles - x_odd * sin_angles
    x_rotated_odd = x_even * sin_angles + x_odd * cos_angles
    
    # 交互に配置して元の形状に戻す
    x_rotated = torch.stack([x_rotated_even, x_rotated_odd], dim=-1)
    x_rotated = x_rotated.flatten(-2)  # 最後の2次元をフラット化
    
    return x_rotated


# --- テスト ---
dim = 64
max_seq_len = 128

angles = precompute_rope_frequencies(dim, max_seq_len)

print("RoPE 周波数テーブル:")
print(f"  形状: {angles.shape}")
print(f"  θ₀ (最低周波数): {1.0 / 10000**(0/dim):.6f}")
print(f"  θ₃₁ (最高周波数): {1.0 / 10000**(62/dim):.6f}")

# テスト入力
batch_size = 2
seq_len = 10
n_heads = 4
x = torch.randn(batch_size, seq_len, n_heads, dim)
x_rotated = apply_rope(x, angles[:seq_len])
print(f"\n入力形状: {x.shape}")
print(f"出力形状: {x_rotated.shape}")
print("✅ RoPE 実装完了")

In [ ]:
# ============================================================
# RoPE の回転パターン可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左: 周波数パターン ---
ax = axes[0]
angles_np = angles.numpy()

# 最初の8次元ペアの角度を位置ごとに表示
for i in range(min(8, dim // 2)):
    ax.plot(range(max_seq_len), angles_np[:, i],
            label=f'dim pair {i}', alpha=0.8, linewidth=1.5)

ax.set_xlabel('位置 m', fontsize=12)
ax.set_ylabel('角度 m·θᵢ (rad)', fontsize=12)
ax.set_title('RoPE: 各次元ペアの回転角度\n（低次元 = 低周波数、高次元 = 高周波数）', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)

# --- 右: 相対位置に対する内積の減衰 ---
ax = axes[1]

# 位置0のベクトルと各位置のベクトルの内積をシミュレーション
torch.manual_seed(42)
q = torch.randn(1, max_seq_len, 1, dim)
q_rotated = apply_rope(q, angles)

# 位置0のベクトルと全位置のベクトルの内積
q0 = q_rotated[0, 0, 0, :]  # 位置0
dot_products = []
for pos in range(max_seq_len):
    q_pos = q_rotated[0, pos, 0, :]
    dot_products.append(torch.dot(q0, q_pos).item())

ax.plot(range(max_seq_len), dot_products, linewidth=2, color='#e74c3c')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('相対位置 (m - 0)', fontsize=12)
ax.set_ylabel('内積 q₀ᵀ · qₘ', fontsize=12)
ax.set_title('RoPE: 位置0との内積\n（相対位置が離れると内積が変化）', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【RoPEの特徴】")
print("・低次元ペアは低周波数 → 大域的な位置情報を捉える")
print("・高次元ペアは高周波数 → 局所的な位置関係を捉える")
print("・内積は相対位置のみに依存するため、外挿が容易")

---

## 5. MHA → MQA → GQA の系譜

### 5.1 3つのAttentionアーキテクチャ

```
【Multi-Head Attention (MHA)】         【Multi-Query Attention (MQA)】
Head 1: Q₁ K₁ V₁                      Head 1: Q₁ K  V
Head 2: Q₂ K₂ V₂                      Head 2: Q₂ K  V
Head 3: Q₃ K₃ V₃                      Head 3: Q₃ K  V
Head 4: Q₄ K₄ V₄                      Head 4: Q₄ K  V
                                       └─ K,V は全ヘッドで共有

【Grouped-Query Attention (GQA)】
Group 1: Q₁ K₁ V₁     ← Head 1,2 が共有
          Q₂ K₁ V₁
Group 2: Q₃ K₂ V₂     ← Head 3,4 が共有
          Q₄ K₂ V₂
└─ 少数のK,Vグループを複数のQヘッドで共有
```

| 手法 | KV ヘッド数 | KV-Cache サイズ | 品質 | 採用例 |
|------|-----------|---------------|------|--------|
| MHA | = Q ヘッド数 | 最大 | 最高 | GPT-3, BERT |
| MQA | 1 | 最小 | やや劣化 | PaLM |
| GQA | Q/グループ数 | 中間 | MHAとほぼ同等 | LLaMA 2, Mistral |

In [ ]:
# ============================================================
# Section 5: GQA（Grouped-Query Attention）の実装
# ============================================================

class MultiHeadAttention(nn.Module):
    """標準的なMulti-Head Attention (MHA)"""
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        # 各ヘッドに独立したQ, K, V射影
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        B, L, _ = x.shape
        # Q, K, V を計算してヘッドに分割
        Q = self.W_q(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        
        # ヘッドを結合
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_o(out)


class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention (GQA)
    
    n_kv_heads < n_heads の場合、複数のQヘッドが同じK,Vグループを共有する。
    n_kv_heads == n_heads → MHA と等価
    n_kv_heads == 1 → MQA と等価
    """
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert n_heads % n_kv_heads == 0, "n_heads は n_kv_heads で割り切れる必要がある"
        
        self.d_model = d_model
        self.n_heads = n_heads          # Qのヘッド数
        self.n_kv_heads = n_kv_heads    # K,Vのヘッド数（グループ数）
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads  # 各K,Vグループを共有するQヘッド数
        
        # Qは全ヘッド分、K,Vはグループ分のみ
        self.W_q = nn.Linear(d_model, n_heads * self.head_dim)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.head_dim)  # 少ないパラメータ！
        self.W_v = nn.Linear(d_model, n_kv_heads * self.head_dim)  # 少ないパラメータ！
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        B, L, _ = x.shape
        
        # Q: (B, L, n_heads, head_dim)
        Q = self.W_q(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        # K, V: (B, L, n_kv_heads, head_dim)
        K = self.W_k(x).view(B, L, self.n_kv_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_kv_heads, self.head_dim).transpose(1, 2)
        
        # K, V をリピートして Q と同じヘッド数にする
        # (B, n_kv_heads, L, head_dim) → (B, n_heads, L, head_dim)
        K = K.repeat_interleave(self.n_rep, dim=1)
        V = V.repeat_interleave(self.n_rep, dim=1)
        
        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_o(out)


# --- パラメータ数の比較 ---
d_model = 4096
n_heads = 32

mha = MultiHeadAttention(d_model, n_heads)
gqa_8 = GroupedQueryAttention(d_model, n_heads, n_kv_heads=8)    # 8グループ
gqa_4 = GroupedQueryAttention(d_model, n_heads, n_kv_heads=4)    # 4グループ
mqa = GroupedQueryAttention(d_model, n_heads, n_kv_heads=1)      # MQA

def count_params(module):
    return sum(p.numel() for p in module.parameters())

print("=" * 60)
print("Attention パラメータ数の比較 (d_model=4096, n_heads=32)")
print("=" * 60)
print(f"  MHA (32 KV heads):  {count_params(mha):>12,} パラメータ")
print(f"  GQA (8 KV heads):   {count_params(gqa_8):>12,} パラメータ")
print(f"  GQA (4 KV heads):   {count_params(gqa_4):>12,} パラメータ")
print(f"  MQA (1 KV head):    {count_params(mqa):>12,} パラメータ")
print(f"\n  MQA は MHA の {count_params(mqa)/count_params(mha)*100:.1f}% のパラメータ数")

In [ ]:
# ============================================================
# MHA/GQA/MQA のKV-Cacheサイズ比較
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

# KV-Cacheサイズの計算
# KV-Cache = 2 * n_kv_heads * head_dim * seq_len * n_layers * batch * sizeof(float16)
n_layers = 32
head_dim = 128  # d_model / n_heads
seq_lengths = [1024, 2048, 4096, 8192, 16384]

configs = [
    ('MHA (32 KV heads)', 32),
    ('GQA (8 KV heads)', 8),
    ('GQA (4 KV heads)', 4),
    ('MQA (1 KV head)', 1),
]

colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

for (name, n_kv), color in zip(configs, colors):
    # バイト数: 2 (K+V) * n_kv * head_dim * seq_len * n_layers * 2 (fp16)
    cache_sizes_gb = [2 * n_kv * head_dim * s * n_layers * 2 / (1024**3) for s in seq_lengths]
    ax.plot(seq_lengths, cache_sizes_gb, 'o-', linewidth=2.5, markersize=8,
            label=name, color=color)

ax.set_xlabel('シーケンス長', fontsize=12)
ax.set_ylabel('KV-Cache サイズ (GB)', fontsize=12)
ax.set_title('KV-Cacheメモリ使用量の比較\n（32層, head_dim=128, FP16, batch=1）', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)

plt.tight_layout()
plt.show()

print("【GQAの効果】")
print("・KV-Cacheサイズを大幅に削減できる")
print("・8グループGQA: MHAの25%のKV-Cache")
print("・品質はMHAとほぼ同等（LLaMA 2で検証済み）")
print("・長いシーケンスほど削減効果が大きい")

---

## 6. KV-Cache（推論高速化）

### 6.1 なぜ KV-Cache が必要か

自己回帰生成では、各ステップで全ての過去トークンの K, V を再計算する必要がある:

```
通常の自己回帰生成（非効率）:
Step 1: [BOS] → K,V計算(1トークン) → 予測
Step 2: [BOS, w₁] → K,V計算(2トークン) → 予測 ← 1トークン分を再計算！
Step 3: [BOS, w₁, w₂] → K,V計算(3トークン) → 予測 ← 2トークン分を再計算！
...

KV-Cacheあり（効率的）:
Step 1: [BOS] → K₁,V₁計算 → cache保存 → 予測
Step 2: [w₁] → K₂,V₂計算 → cache追加 → concat(K₁₋₂, V₁₋₂) → 予測
Step 3: [w₂] → K₃,V₃計算 → cache追加 → concat(K₁₋₃, V₁₋₃) → 予測
→ 新しいトークンのK,Vだけ計算すればOK！
```

**計算量の削減**: O(n²) → O(n)（各ステップあたり）

In [ ]:
# ============================================================
# Section 6: KV-Cache の実装と効果の検証
# ============================================================

class CausalSelfAttentionWithCache(nn.Module):
    """KV-Cache付きのCausal Self-Attention"""
    
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, kv_cache=None):
        """
        Parameters:
            x: (B, L, d_model) — L=1の場合はKV-Cacheを使った推論
            kv_cache: (past_K, past_V) のタプル、またはNone
        Returns:
            output, (new_K, new_V)
        """
        B, L, _ = x.shape
        
        Q = self.W_q(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        
        # KV-Cacheがあれば結合
        if kv_cache is not None:
            past_K, past_V = kv_cache
            K = torch.cat([past_K, K], dim=2)  # シーケンス次元で結合
            V = torch.cat([past_V, V], dim=2)
        
        # 新しいキャッシュを保存
        new_cache = (K.detach(), V.detach())
        
        # Attention計算
        seq_len_k = K.shape[2]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal mask（新しいトークンは過去の全トークンを見れる）
        if L > 1:
            causal_mask = torch.triu(torch.ones(L, seq_len_k), diagonal=1).bool()
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        
        return self.W_o(out), new_cache


# --- KV-Cacheなし vs ありの比較 ---
import time

d_model_small = 256
n_heads_small = 4
attn_module = CausalSelfAttentionWithCache(d_model_small, n_heads_small)
attn_module.eval()

# シミュレーション: 50トークンを自己回帰で生成
gen_length = 50

# --- KV-Cacheなし ---
torch.manual_seed(42)
start = time.time()
with torch.no_grad():
    for t in range(1, gen_length + 1):
        # 毎回全シーケンスを再計算
        x = torch.randn(1, t, d_model_small)
        out, _ = attn_module(x, kv_cache=None)
time_no_cache = time.time() - start

# --- KV-Cacheあり ---
torch.manual_seed(42)
start = time.time()
with torch.no_grad():
    cache = None
    for t in range(gen_length):
        # 新しいトークンだけを処理
        x = torch.randn(1, 1, d_model_small)
        out, cache = attn_module(x, kv_cache=cache)
time_with_cache = time.time() - start

print("=" * 60)
print(f"KV-Cache の効果（{gen_length}トークン生成）")
print("=" * 60)
print(f"  Cache なし: {time_no_cache:.4f} 秒")
print(f"  Cache あり: {time_with_cache:.4f} 秒")
print(f"  高速化率:   {time_no_cache / time_with_cache:.2f}x")

In [ ]:
# ============================================================
# KV-Cacheの計算量削減を可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左: ステップごとの計算量 ---
ax = axes[0]
steps = np.arange(1, 101)

# Cacheなし: 各ステップでt²の計算（Attention行列のサイズ）
flops_no_cache = steps ** 2
# Cacheあり: 各ステップでtの計算（新しいトークン × 過去のK）
flops_with_cache = steps

ax.plot(steps, flops_no_cache, linewidth=2.5, label='KV-Cache なし (O(t²))',
        color='#e74c3c')
ax.plot(steps, flops_with_cache, linewidth=2.5, label='KV-Cache あり (O(t))',
        color='#2ecc71')
ax.fill_between(steps, flops_with_cache, flops_no_cache, alpha=0.1, color='#e74c3c')
ax.set_xlabel('生成ステップ t', fontsize=12)
ax.set_ylabel('相対計算量', fontsize=12)
ax.set_title('ステップごとの計算量比較', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# --- 右: 累積計算量 ---
ax = axes[1]
cumulative_no_cache = np.cumsum(flops_no_cache)
cumulative_with_cache = np.cumsum(flops_with_cache)

ax.plot(steps, cumulative_no_cache, linewidth=2.5, label='KV-Cache なし (O(t³))',
        color='#e74c3c')
ax.plot(steps, cumulative_with_cache, linewidth=2.5, label='KV-Cache あり (O(t²))',
        color='#2ecc71')
ax.set_xlabel('生成ステップ t', fontsize=12)
ax.set_ylabel('累積計算量', fontsize=12)
ax.set_title('累積計算量の比較\n（差は生成長に応じて拡大）', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【KV-Cacheのトレードオフ】")
print("・計算量: O(t²) → O(t) に削減（各ステップ）")
print("・メモリ: 過去のK,Vを保存する追加メモリが必要")
print("・GQAと組み合わせてメモリ削減が重要")

---

## 7. FlashAttention（概念）

### 7.1 標準Attentionのメモリ問題

標準的なAttention計算:
1. $S = QK^T$ → $O(n^2)$ のメモリ（Attention行列）
2. $P = \text{softmax}(S)$ → $O(n^2)$ のメモリ
3. $O = PV$ → $O(nd)$ のメモリ

シーケンス長 $n = 8192$, ヘッド数 32 の場合:
- Attention行列: $32 \times 8192 \times 8192 \times 2$ bytes $\approx$ **4 GB** (FP16)

### 7.2 FlashAttentionの解決策

**核心アイデア**: Attention行列を**一度にメモリに載せない**。タイリング（ブロック分割）で計算し、SRAM（高速メモリ）上で処理する。

```
【標準Attention】                    【FlashAttention】
    Q  ×  K^T  →  S (n×n)              Q  ×  K^T （ブロックごと）
                   ↓                         ↓
              softmax(S)                softmax（オンラインで累積）
                   ↓                         ↓
              S  ×  V  →  O            ブロック結果を累積 → O
                                        
メモリ: O(n²)                      メモリ: O(n) ← 大幅削減！
IO: HBM ←→ SRAM 多数回             IO: 少数回 ← 高速！
```

### 7.3 IO-Awareness

- **HBM (High Bandwidth Memory)**: 大容量だがアクセスが遅い
- **SRAM (Static RAM)**: 小容量だがアクセスが速い
- FlashAttentionはSRAMに収まるブロックサイズで計算することで、HBMアクセスを最小化

| 手法 | メモリ | IO回数 | 実装 |
|------|--------|--------|------|
| 標準Attention | O(n²) | 多い | PyTorch標準 |
| FlashAttention | O(n) | 少ない | CUDA kernel |
| FlashAttention-2 | O(n) | さらに少ない | 改良版 |

---

## 8. RMSNorm

In [ ]:
# ============================================================
# Section 8: RMSNorm の実装と比較
# ============================================================

class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization
    
    標準的な Layer Norm と異なり、平均を引かない（中心化しない）。
    RMSNorm(x) = x / RMS(x) * γ
    RMS(x) = sqrt(mean(x²) + ε)
    
    利点:
    - Layer Normより計算が単純（平均の計算を省略）
    - 実験的にLayer Normと同等の性能
    - LLaMA, Mistral 等で採用
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # γ（学習可能スケール）
    
    def forward(self, x):
        # RMS(x) = sqrt(mean(x²) + ε)
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        # 正規化してスケール
        return (x / rms) * self.weight


# --- Layer Norm との比較 ---
torch.manual_seed(42)
x = torch.randn(2, 10, 256)  # (batch, seq_len, d_model)

layer_norm = nn.LayerNorm(256)
rms_norm = RMSNorm(256)

out_ln = layer_norm(x)
out_rms = rms_norm(x)

print("=" * 60)
print("RMSNorm vs LayerNorm の比較")
print("=" * 60)
print(f"\n入力:")
print(f"  平均: {x.mean().item():.4f}, 標準偏差: {x.std().item():.4f}")
print(f"\nLayerNorm出力:")
print(f"  平均: {out_ln.mean().item():.6f}, 標準偏差: {out_ln.std().item():.4f}")
print(f"  → 平均≈0, 分散≈1 に正規化")
print(f"\nRMSNorm出力:")
print(f"  平均: {out_rms.mean().item():.4f}, 標準偏差: {out_rms.std().item():.4f}")
print(f"  → 平均は0に強制しない、スケールのみ正規化")

# パラメータ数の比較
print(f"\nパラメータ数:")
print(f"  LayerNorm: {sum(p.numel() for p in layer_norm.parameters())} (γ + β)")
print(f"  RMSNorm:   {sum(p.numel() for p in rms_norm.parameters())} (γ のみ)")

---

## 9. SwiGLU

In [ ]:
# ============================================================
# Section 9: SwiGLU の実装と活性化関数の比較
# ============================================================

class StandardFFN(nn.Module):
    """標準的な Feed-Forward Network (Transformer原論文)
    FFN(x) = ReLU(xW₁ + b₁)W₂ + b₂
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))


class SwiGLUFFN(nn.Module):
    """SwiGLU Feed-Forward Network (LLaMA, PaLM等で採用)
    SwiGLU(x) = (Swish(xW₁) ⊙ xW₃) W₂
    
    Swish(x) = x · σ(x)
    
    ゲート機構により、情報の流れを制御する。
    パラメータ数は増えるが、性能向上が確認されている。
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # ゲート
        self.w3 = nn.Linear(d_model, d_ff, bias=False)  # 値
        self.w2 = nn.Linear(d_ff, d_model, bias=False)  # 出力射影
    
    def forward(self, x):
        # Swish(xW₁) ⊙ xW₃
        swish_gate = F.silu(self.w1(x))  # SiLU = Swish with β=1
        value = self.w3(x)
        return self.w2(swish_gate * value)


# --- 活性化関数の比較可視化 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_range = torch.linspace(-5, 5, 200)

# --- 左: 活性化関数の比較 ---
ax = axes[0]
ax.plot(x_range.numpy(), F.relu(x_range).numpy(), linewidth=2, label='ReLU', color='#e74c3c')
ax.plot(x_range.numpy(), F.gelu(x_range).numpy(), linewidth=2, label='GELU', color='#3498db')
ax.plot(x_range.numpy(), F.silu(x_range).numpy(), linewidth=2, label='SiLU (Swish)', color='#2ecc71')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('活性化関数の比較\nReLU vs GELU vs SiLU/Swish', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)

# --- 右: FFN パラメータ数の比較 ---
ax = axes[1]
d_model_test = 4096
d_ff_standard = d_model_test * 4          # 標準: 4x
d_ff_swiglu = int(d_model_test * 4 * 2/3) # SwiGLU: 調整して同等のパラメータ数

standard_ffn = StandardFFN(d_model_test, d_ff_standard)
swiglu_ffn = SwiGLUFFN(d_model_test, d_ff_swiglu)

params_standard = sum(p.numel() for p in standard_ffn.parameters())
params_swiglu = sum(p.numel() for p in swiglu_ffn.parameters())

bars = ax.bar(['Standard FFN\n(ReLU, d_ff=4d)', 'SwiGLU FFN\n(Swish, d_ff=8d/3)'],
              [params_standard / 1e6, params_swiglu / 1e6],
              color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, [params_standard, params_swiglu]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val/1e6:.1f}M', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('パラメータ数 (M)', fontsize=12)
ax.set_title(f'FFN パラメータ数の比較\n(d_model={d_model_test})', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("【SwiGLUの利点】")
print("・ゲート機構により情報の流れを制御」")
print("・ReLUの『死んだニューロン』問題を回避")
print("・d_ffを2/3に調整してパラメータ数をほぼ同等に")
print("・LLaMA, PaLM, Mistral 等で採用")

---

## 10. モデル並列化の概要

大規模LLMは単一GPUに載りきらないため、複数GPUに分散して計算します。

### 10.1 データ並列 (Data Parallel)

```
各GPUに同じモデルのコピー:

GPU 0: Model Copy 0 ← Batch 0
GPU 1: Model Copy 1 ← Batch 1
GPU 2: Model Copy 2 ← Batch 2
GPU 3: Model Copy 3 ← Batch 3
              ↓
        勾配を平均 (AllReduce)
              ↓
        パラメータ更新
```

- **利点**: 実装が簡単
- **制約**: 各GPUにモデル全体が載る必要がある

### 10.2 テンソル並列 (Tensor Parallel)

```
1つの層を複数GPUに分割:

              x
              ↓
    ┌─────────┴─────────┐
    │                   │
  W₁[:, :d/2]      W₁[:, d/2:]
  (GPU 0)           (GPU 1)
    │                   │
    └─────────┬─────────┘
              ↓
          AllReduce
```

- **利点**: 超大型モデルも扱える
- **制約**: GPU間通信のオーバーヘッド

### 10.3 パイプライン並列 (Pipeline Parallel)

```
モデルの層を異なるGPUに配置:

GPU 0: Layer 1-8
  ↓
GPU 1: Layer 9-16
  ↓
GPU 2: Layer 17-24
  ↓
GPU 3: Layer 25-32
```

- **利点**: メモリ効率が良い
- **制約**: パイプラインバブル（GPU待機時間）

### 10.4 比較表

| 手法 | 分割対象 | 通信量 | 実装難度 | 適用場面 |
|------|---------|--------|---------|--------|
| データ並列 | バッチ | 勾配 | 低 | モデルが1GPUに載る場合 |
| テンソル並列 | 層の内部 | 活性化 | 高 | 非常に大きな層 |
| パイプライン並列 | 層間 | 活性化 | 中 | 深いモデル |
| ZeRO (DeepSpeed) | パラメータ+勾配+状態 | 必要時に取得 | 中 | 汎用 |

---

## 11. 現代LLMアーキテクチャの統合

### 11.1 LLaMA-styleモデルの全体構成

LLaMA（Meta, 2023）は現代LLMのデファクトスタンダードとなっており、多くのオープンモデルがこのアーキテクチャを採用しています。

```
入力トークン ID
    ↓
[Token Embedding]  ← 語彙サイズ × d_model
    ↓
┌──────────────────────────────────┐
│ Decoder Block × N_layers          │
│                                    │
│   ├─ RMSNorm (Pre-Norm)           │  ← Layer Normの代わり
│   ├─ GQA (Grouped-Query Attention) │  ← MHAの代わり
│   │   └─ + RoPE                   │  ← Sinusoidal PEの代わり
│   ├─ Residual Connection          │
│   │                                │
│   ├─ RMSNorm (Pre-Norm)           │
│   ├─ SwiGLU FFN                   │  ← ReLU FFNの代わり
│   └─ Residual Connection          │
│                                    │
└──────────────────────────────────┘
    ↓
[RMSNorm]
    ↓
[Linear → vocab_size]  ← 次トークン予測
```

In [ ]:
# ============================================================
# Section 11: LLaMA-style モデルの統合実装
# ============================================================

class LLaMABlock(nn.Module):
    """LLaMA-style Transformer ブロック
    
    コンポーネント:
    - RMSNorm (Pre-Norm)
    - Grouped-Query Attention with RoPE
    - SwiGLU FFN
    - Residual Connections
    """
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff):
        super().__init__()
        # Pre-Norm (RMSNorm)
        self.attn_norm = RMSNorm(d_model)
        self.ffn_norm = RMSNorm(d_model)
        
        # GQA
        self.attention = GroupedQueryAttention(d_model, n_heads, n_kv_heads)
        
        # SwiGLU FFN
        self.ffn = SwiGLUFFN(d_model, d_ff)
    
    def forward(self, x, mask=None):
        # Pre-Norm → Attention → Residual
        h = x + self.attention(self.attn_norm(x), mask)
        # Pre-Norm → FFN → Residual
        out = h + self.ffn(self.ffn_norm(h))
        return out


class MiniLLaMA(nn.Module):
    """最小限のLLaMA-styleモデル"""
    def __init__(self, vocab_size, d_model, n_heads, n_kv_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        # トークン埋め込み（位置埋め込みはRoPEで代替）
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        
        # Transformer ブロック
        self.blocks = nn.ModuleList([
            LLaMABlock(d_model, n_heads, n_kv_heads, d_ff)
            for _ in range(n_layers)
        ])
        
        # 出力
        self.norm = RMSNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size, bias=False)
        
        # RoPE周波数（事前計算）
        self.rope_angles = precompute_rope_frequencies(d_model // n_heads, max_seq_len)
    
    def forward(self, x, mask=None):
        h = self.tok_emb(x)
        # 注: 簡略化のためRoPEの適用はAttention内部で行うべきだが
        #     ここではブロック呼び出しのみ示す
        for block in self.blocks:
            h = block(h, mask)
        h = self.norm(h)
        logits = self.output(h)
        return logits


# --- モデル構成の例 ---
configs = {
    'Mini (教育用)': dict(vocab_size=1000, d_model=256, n_heads=4, n_kv_heads=2,
                          n_layers=4, d_ff=683, max_seq_len=128),
    'LLaMA-7B相当': dict(vocab_size=32000, d_model=4096, n_heads=32, n_kv_heads=32,
                          n_layers=32, d_ff=11008, max_seq_len=2048),
    'LLaMA2-7B相当': dict(vocab_size=32000, d_model=4096, n_heads=32, n_kv_heads=8,
                           n_layers=32, d_ff=11008, max_seq_len=4096),
}

print("=" * 70)
print("LLaMA-style モデル構成の比較")
print("=" * 70)
print(f"{'構成':<20} {'d_model':<10} {'heads':<8} {'KV heads':<10} {'layers':<8} {'params':>12}")
print("-" * 70)

for name, cfg in configs.items():
    model = MiniLLaMA(**cfg)
    n_params = sum(p.numel() for p in model.parameters())
    
    if n_params >= 1e9:
        param_str = f"{n_params/1e9:.1f}B"
    elif n_params >= 1e6:
        param_str = f"{n_params/1e6:.1f}M"
    else:
        param_str = f"{n_params/1e3:.0f}K"
    
    print(f"{name:<20} {cfg['d_model']:<10} {cfg['n_heads']:<8} {cfg['n_kv_heads']:<10} {cfg['n_layers']:<8} {param_str:>12}")
    del model

print("\n✅ 現代LLMのコンポーネントをすべて統合したモデルを構築")
print("   RMSNorm + GQA + RoPE + SwiGLU = LLaMA-style Architecture")

---

## 12. まとめ・チートシート・よくある間違い・自己評価クイズ

### 12.1 まとめ

このノートブックでは、GPTから現代LLMへの技術進化を学びました。

1. **Scaling Laws**: モデル性能はパラメータ数・データ量のべき乗則に従う（Chinchilla: D≈20N）
2. **RoPE**: 回転行列による相対位置エンコーディング → 長さ外挿が可能
3. **GQA**: K,Vヘッドを共有してKV-Cacheのメモリを削減
4. **KV-Cache**: 推論時にK,Vを再利用して計算量をO(t²)→O(t)に削減
5. **FlashAttention**: タイリングによるメモリ効率化（O(n²)→O(n)）
6. **RMSNorm**: LayerNormの簡略版（平均を省略）
7. **SwiGLU**: ゲート機構付きFFN（ReLUより高性能）
8. **モデル並列化**: データ/テンソル/パイプライン並列で大規模モデルを訓練

### 12.2 技術進化チートシート

| コンポーネント | 旧来 | 現代 | 効果 |
|-------------|------|------|------|
| 位置エンコーディング | Sinusoidal / Learned | RoPE | 相対位置、長さ外挿 |
| Attention | MHA | GQA | KV-Cacheメモリ削減 |
| 正規化 | LayerNorm (Post-LN) | RMSNorm (Pre-LN) | 計算効率、訓練安定性 |
| FFN | ReLU FFN | SwiGLU | 性能向上 |
| 推論最適化 | なし | KV-Cache + FlashAttention | 速度・メモリ |
| 学習戦略 | 大モデル重視 | Chinchilla最適 | 計算効率 |

### 12.3 よくある間違い

#### 間違い 1: RoPEはEmbeddingに加算するものだと思い込む

Sinusoidal PEは入力埋め込みに**加算**しますが、RoPEはQ, Kベクトルに**回転を適用**します。Vには適用しません。これにより、内積が自然に相対位置に依存するようになります。

#### 間違い 2: GQAとMQAを混同する

MQA（Multi-Query Attention）はK,Vを**1組だけ**使い全ヘッドで共有します。GQA（Grouped-Query Attention）は**複数グループ**のK,Vを使い、各グループを複数のQヘッドで共有します。GQAはMHAとMQAの中間であり、品質と効率のバランスが優れています。

#### 間違い 3: KV-Cacheは訓練時にも使えると思う

KV-Cacheは**推論時の自己回帰生成**専用の最適化です。訓練時はTeacher Forcingで全トークンを一度に処理するため、KV-Cacheは不要です。

### 12.4 自己評価クイズ

---

**Q1.** Chinchilla則によると、パラメータ数Nに対して最適なトークン数Dはどのくらいですか？

<details>
<summary>💡 答えを見る</summary>

**答え**: D ≈ 20N

Chinchilla (Hoffmann et al., 2022) によると、計算予算が固定の場合、パラメータ数とトークン数を同じ比率で増やすべきです。具体的には、トークン数はパラメータ数の約20倍が最適。例えば、7Bモデルなら約140Bトークンが最適です。

</details>

---

**Q2.** RoPEが Sinusoidal PE よりも長いシーケンスに対応しやすい理由は？

<details>
<summary>💡 答えを見る</summary>

**答え**: RoPEは**相対位置**のみに依存するため。

Sinusoidal PEは絶対位置をベクトルに加算するため、訓練時の最大長を超えると未知の位置情報になります。一方、RoPEはQ·Kの内積が相対位置(m-n)のみに依存するように設計されているため、訓練時に見なかった長さでもある程度外挿が可能です。

</details>

---

**Q3.** GQA で n_heads=32, n_kv_heads=8 の場合、各KVグループを何個のQヘッドが共有しますか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 4個

32 ÷ 8 = 4 なので、各K,Vグループを4つのQヘッドが共有します。これにより、KV-Cacheのサイズはフルのn_heads=32の場合の8/32 = 25%に削減されます。

</details>

---

**Q4.** KV-Cacheを使わない場合、100トークンの生成にかかるAttentionの総計算量のオーダーは？

<details>
<summary>💡 答えを見る</summary>

**答え**: O(n³)

KV-Cacheなしの場合、ステップtでのAttention計算はO(t²)（t×tのAttention行列）。これを1からnまで合計すると、Σt²≈n³/3 = O(n³)。KV-Cacheありなら各ステップはO(t)、合計O(n²)に削減されます。

</details>

---

**Q5.** SwiGLUのFFNでd_ffを標準FFNの2/3に設定する理由は？

<details>
<summary>💡 答えを見る</summary>

**答え**: パラメータ数を同等に保つため。

SwiGLUは3つの重み行列（W₁, W₃, W₂）を使うため、標準FFN（W₁, W₂の2つ）より多くのパラメータが必要です。d_ffを2/3に調整することで、総パラメータ数を標準FFNと同等に保ちながら、ゲート機構の利点を得られます。

</details>

---

### 次のステップ

次の **Notebook 167: ファインチューニングとアライメント** では、これらの現代LLMアーキテクチャの上に構築される応用技術を学びます。

- LoRA/QLoRA による効率的なファインチューニング
- Instruction Tuning と RLHF
- RAG パイプラインの完成
- LLM の評価指標

---

## 参考文献

1. Kaplan, J., et al. (2020). *Scaling Laws for Neural Language Models*. arXiv:2001.08361.
2. Hoffmann, J., et al. (2022). *Training Compute-Optimal Large Language Models* (Chinchilla). arXiv:2203.15556.
3. Su, J., et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding*. arXiv:2104.09864.
4. Ainslie, J., et al. (2023). *GQA: Training Generalized Multi-Query Transformer Models*. arXiv:2305.13245.
5. Dao, T., et al. (2022). *FlashAttention: Fast and Memory-Efficient Exact Attention*. NeurIPS 2022.
6. Zhang, B. & Sennrich, R. (2019). *Root Mean Square Layer Normalization*. NeurIPS 2019.
7. Shazeer, N. (2020). *GLU Variants Improve Transformer*. arXiv:2002.05202.
8. Touvron, H., et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971.